The purpose of this notebook is to check the analysis rendered by Claude Code is accurate. 

# Setup

In [19]:
import pandas as pd
import numpy as np
from utils import load_all_adl
from pycm import ConfusionMatrix

In [20]:
adl_df = load_all_adl()

## Utility Functions

In [59]:
def p05(x):
    return x.quantile(0.05)
def p10(x):
    return x.quantile(0.1)
def p90(x):
    return x.quantile(0.9)
def p95(x):
    return x.quantile(0.95)

def print_cm_table(cm):
    """Print out the metrics we care about from the confusion matrix"""
    cm.print_matrix()
    
    classes = list(cm.classes)
    metrics = {
        "True Positive Rate": cm.TPR,
        "False Positive Rate": cm.FPR,
        "Specificity":        cm.TNR,
        "Precision":        cm.PPV,  # same as TPR
    }

    col_w = 12
    label_w = 22

    # Header
    print(f"{'':>{label_w}}", end="")
    for cls in classes:
        print(f"{str(cls):>{col_w}}", end="")
    print()

    # Rows
    for metric_name, metric_dict in metrics.items():
        print(f"{metric_name:>{label_w}}", end="")
        for cls in classes:
            val = metric_dict[cls]
            formatted = f"{val * 100:.1f}%" if val is not None else "N/A"
            print(f"{formatted:>{col_w}}", end="")
        print()

# Prompt 3
Here I asked the agent to figure out an explainable decision process for figuring out locomotion status with access to the ground truth labels.
To determine locomotion status, the agent constructed a series of one-vs-rest tests as follows: 

IF HIP_accY < 500 → LIE

ELSE IF rolling_std(SHOE_AngVelMagnitude, 1sec) > 700 → WALK

ELSE IF RKN^_accZ > 600 → SIT

ELSE → STAND

In [24]:
# Functions unique to analyzing prompt 3 

# Need this to check the logic for the 'Walk' condition
def rolling_1sec_std(df, col='InertialMeasurementUnit_L-SHOE_AngVelBodyFrameX'):
    """
    Compute a rolling 1-second standard deviation of `col` for each subject
    independently, using MILLISEC (ms) as the time axis.

    Returns a Series aligned to df's index.
    """
    def _per_subject(group):
        time_idx = pd.to_timedelta(group['MILLISEC'], unit='ms')
        return (
            group[col]
            .set_axis(time_idx)
            .rolling('1000ms')
            .std()
            .set_axis(group.index)  # restore original integer index
        )

    return df.groupby('recording', group_keys=False).apply(_per_subject)

adl_df['rolling_std'] = rolling_1sec_std(adl_df)

In [25]:
### Lie vs. Rest 

# Medians/5th/95th percentiles match what was summarized by agent
adl_df.groupby('Locomotion')['Accelerometer_HIP_accY'].agg(['median', p05, p95])

# Agent summary claimed that this test was 95.2% sensitive, <1% FPR for Sit/Walk, and ~5% FPR for stand. Is this true? 
adl_df['lie_test'] = adl_df['Accelerometer_HIP_accY'] < 500
lie_test_df = adl_df[['lie_test', 'Locomotion']].groupby('Locomotion')['lie_test'].agg(['count', 'sum'])
tp = lie_test_df.at['Lie', 'sum']
# Sensitivity (TP / (TP + FN)) is actually 91.7%
print(f'Sensitivity: {tp / (lie_test_df.at["Lie", "count"]):.3f}')
fpr_sit = lie_test_df.at['Sit', 'sum'] / lie_test_df.at['Sit', 'count']
fpr_walk = lie_test_df.at['Walk', 'sum'] / lie_test_df.at['Walk', 'count']
fpr_stand = lie_test_df.at['Stand', 'sum'] / lie_test_df.at['Stand', 'count']
# FPRs match what was given
print(f'FPR Sit:     {fpr_sit:.3f}')
print(f'FPR Walk:    {fpr_walk:.3f}')
print(f'FPR Stand:   {fpr_stand:.3f}')

Sensitivity: 0.917
FPR Sit:     0.005
FPR Walk:    0.008
FPR Stand:   0.051


In [26]:
### Walk vs. Rest

# Verifying mean STD over rolling 1 second window; code was ambiguous about to use X or Y field for shoe angular velocity sensor. 
# In either case, these values don't exactly match up with values in summary.
# Using the X axis since that seems to have bigger difference between walk and non-walk classes
adl_df.groupby('Locomotion')['rolling_std'].mean()

# Test proposed use of 500 cutoff 
adl_df['walk_test'] = adl_df['rolling_std'] > 500
# Claims sensitivity 95% at 500 threshold
walk_test_df = adl_df[['walk_test', 'Locomotion']].groupby('Locomotion')['walk_test'].agg(['count', 'sum'])
tp = walk_test_df.at['Walk', 'sum']
# Actual sensitivity is 90.2%
print(f'Sensitivity: {tp / walk_test_df.at["Walk", "count"]:.3f}')
# Claims FP rate form stand of 18% - actual result is closer to 20% 
fpr_sit = walk_test_df.at['Sit', 'sum'] / walk_test_df.at['Sit', 'count']
fpr_lie = walk_test_df.at['Lie', 'sum'] / walk_test_df.at['Lie', 'count']
fpr_stand = walk_test_df.at['Stand', 'sum'] / walk_test_df.at['Stand', 'count']
print(f'FPR Sit:     {fpr_sit:.3f}')
print(f'FPR Lie:     {fpr_lie:.3f}')
print(f'FPR Stand:   {fpr_stand:.3f}')

Sensitivity: 0.902
FPR Sit:     0.040
FPR Lie:     0.108
FPR Stand:   0.198


In [27]:
### Standing vs. Rest 

# Verifying medians in summary - they are
adl_df.groupby('Locomotion')['Accelerometer_RKN^_accZ'].agg(['median', p10, p90])

# Verifying asserted sensitivity with 600 threshold
adl_df['sit_test'] = adl_df['Accelerometer_RKN^_accZ'] > 600
sit_test_df = adl_df[['sit_test', 'Locomotion']].groupby('Locomotion')['sit_test'].agg(['count', 'sum'])
tp = sit_test_df.at['Sit', 'sum']
# Asserted sensitivity was 97.7%, actual sensitivity is 95.6%
print(f'Sensitivity: {tp / sit_test_df.at["Sit", "count"]:.3f}')
fpr_walk = sit_test_df.at['Walk', 'sum'] / sit_test_df.at['Sit', 'count']
fpr_lie = sit_test_df.at['Lie', 'sum'] / sit_test_df.at['Lie', 'count']
fpr_stand = sit_test_df.at['Stand', 'sum'] / sit_test_df.at['Stand', 'count']
# Asserted FPR for stand and walk "combined" (no explanation) was 14% - actual is 16.1% for walk and 13% for stand
print(f'FPR Walk:    {fpr_walk:.3f}')
print(f'FPR Lie:     {fpr_lie:.3f}')
print(f'FPR Stand:   {fpr_stand:.3f}')

Sensitivity: 0.956
FPR Walk:    0.161
FPR Lie:     0.229
FPR Stand:   0.130


In [61]:
### Overall logic
conditions = [
    adl_df['Accelerometer_HIP_accY'] < 500,
    adl_df['rolling_std'] > 700, 
    adl_df['Accelerometer_RKN^_accZ'] > 600
]
choices = ['Lie', 'Walk', 'Sit']
adl_df['predicted_Locomotion'] = np.select(conditions, choices, default='Stand')
overall_test_df = adl_df.loc[adl_df['Locomotion'].notna()]
y_true = overall_test_df['Locomotion']
y_pred = overall_test_df['predicted_Locomotion']
cm = ConfusionMatrix(actual_vector = overall_test_df['Locomotion'].to_list(),
                     predict_vector = overall_test_df['predicted_Locomotion'].to_list())

# TPR > 90 for Lie and Sit, 73% for Stand, 84% for Walk
# Sensitivity > 90 for all classes
# Pretty good decision algorithm for being that simple 
print_cm_table(cm)

Predict      Lie          Sit          Stand        Walk         
Actual
Lie          23287        562          1021         525          

Sit          590          110372       4018         3088         

Stand        11461        20240        163032       28004        

Walk         1013         2294         17633        110056       


                               Lie         Sit       Stand        Walk
    True Positive Rate       91.7%       93.5%       73.2%       84.0%
   False Positive Rate        2.8%        6.1%        8.3%        8.6%
           Specificity       97.2%       93.9%       91.7%       91.4%
             Precision       64.1%       82.7%       87.8%       77.7%


# Prompt 7
Here I'm asking Claude Code to predict locomotive status without access to the ground truth labels, based on a description of the dataset that I gave it

In [62]:
# Load in actual and predicted locomotion labels
adl_df
pred_df = pd.read_parquet('prompt_7/labels.parquet')
true_df = adl_df[['MILLISEC', 'recording', 'Locomotion']]
pred_df = pred_df[['MILLISEC', 'recording', 'label']]

comp_df = true_df.merge(pred_df, on=['recording', 'MILLISEC']).rename(columns={'Locomotion': 'Locomotion_actual', 'label': 'Locomotion_predicted'})
comp_df = comp_df.dropna(subset=['Locomotion_actual'])

In [63]:
cm = ConfusionMatrix(actual_vector = comp_df['Locomotion_actual'].to_list(),
                     predict_vector = comp_df['Locomotion_predicted'].to_list())

print_cm_table(cm)

Predict      Lie          Sit          Stand        Walk         
Actual
Lie          12696        11851        416          432          

Sit          0            114772       1642         1654         

Stand        6889         5088         171522       39238        

Walk         2            2223         10442        118329       


                               Lie         Sit       Stand        Walk
    True Positive Rate       50.0%       97.2%       77.0%       90.3%
   False Positive Rate        1.5%        5.1%        4.6%       11.3%
           Specificity       98.5%       94.9%       95.4%       88.7%
             Precision       64.8%       85.7%       93.2%       74.1%


## Summary
With the exception of the 'Lie' category, the unsupervised approach dominated the supervised approach for learning to distinguish sitting and standing, did comparably but not quite as well in identifying walking, and did significantly worse in identifying lying down. While this is surprising, it also came up with a more sophisticated approach than the very simple rule-based approach that prompt 3 used. Most of the 'Lie' states that the unsupervised approach missed were sitting - it had fewer miscategorizations as standing or walking. 

# Pipeline 1

In [31]:
df = pd.read_parquet('secrets/all_ADL.parquet')

In [28]:
# Dataset is sampled at 30 Hz; each row = 1/30 seconds
SAMPLE_RATE_HZ = 30

# Fill Nulls 
# Drop rows not in these states (in this case just removes NaNs)
loco_states = ['Lie', 'Walk', 'Sit', 'Stand', 'Uncategorized']
df['Locomotion'] = df['Locomotion'].fillna('Uncategorized')
loco_df = df[df['Locomotion'].isin(loco_states)]

time_per_state = (
    loco_df.groupby(['subject', 'Locomotion'])
    .size()
    .div(SAMPLE_RATE_HZ)          # convert rows -> seconds
    .div(60)                       # seconds -> minutes
    .rename('minutes')
    .reset_index()
    .pivot(index='subject', columns='Locomotion', values='minutes')
    [loco_states]                  # consistent column order
    .round(1)
)

time_per_state['Total'] = time_per_state.sum(axis=1)

print("Time per subject per Locomotion state (minutes):")
print(time_per_state.to_string())

Time per subject per Locomotion state (minutes):
Locomotion  Lie  Walk   Sit  Stand  Uncategorized  Total
subject                                                 
S1          3.4  17.1  19.5   34.9           25.0   99.9
S2          2.7  19.4  22.4   29.4           21.4   95.3
S3          5.1  19.9  13.0   31.1           12.0   81.1
S4          3.0  16.4  10.8   28.3           23.4   81.9


# Pipeline 9
Here I'm trying the two-step process of first generating a plan, and then executing that on the file.

In [21]:
true_df = adl_df
pred_df = pd.read_parquet('pipeline/CC_pipelines/pipeline_9b_output_file.parquet')

In [60]:
comp_df = true_df[['MILLISEC', 'recording', 'Locomotion']].merge(pred_df[['MILLISEC', 'recording', 'posture']], on=['MILLISEC', 'recording'])
comp_df = comp_df.rename(columns={'Locomotion': 'true', 'posture': 'predicted'})
# Restrict analysis to only those frames for which we have ground-truth labels
comp_df = comp_df.loc[comp_df['true'].notna()]
# Generated code uses gerands, so replace those 
comp_df['predicted'] = comp_df['predicted'].replace({'Standing': 'Stand', 
                                                     'Sitting': 'Sit', 
                                                     'Walking': 'Walk', 
                                                     'Lying': 'Lie'})

cm = ConfusionMatrix(actual_vector = comp_df['true'].to_list(),
                     predict_vector = comp_df['predicted'].to_list())

print_cm_table(cm)

Predict      Lie          Sit          Stand        Walk         
Actual
Lie          16786        8406         203          0            

Sit          519          109227       8322         0            

Stand        19960        3551         198385       841          

Walk         2212         1201         102195       25388        


                               Lie         Sit       Stand        Walk
    True Positive Rate       66.1%       92.5%       89.1%       19.4%
   False Positive Rate        4.8%        3.5%       40.3%        0.2%
           Specificity       95.2%       96.5%       59.7%       99.8%
             Precision       42.5%       89.2%       64.2%       96.8%
